Usage of PLINK2: https://www.cog-genomics.org/plink/2.0/
EDA on genotype VCF & VCF to PLINK: https://github.com/rl3328/xqtl-analysis/discussions/31
missing on genotype VCF: https://github.com/rl3328/xqtl-analysis/discussions/33

### 📑 **VCF File Inspection and Conversion Workflow**

---

### 🛠️ **1. Inspecting the VCF File**

#### **1.1 Chromosome names and contigs**
Check the chromosome naming convention and list of contigs.
```bash
bcftools view -h your_file.vcf.gz | grep "##contig"
```

#### **1.2 Number of samples**
Count the number of samples in the VCF.
```bash
bcftools query -l your_file.vcf.gz | wc -l
```

#### **1.3 Number of variants**
Get the total number of variants.
```bash
bcftools index -n your_file.vcf.gz
```

#### **1.4 Variant types present**
Identify different types of variants (SNPs, indels, etc.).
```bash
bcftools view -H your_file.vcf.gz | cut -f4,5 | sort | uniq -c
```

#### **1.5 Allele frequency distribution**
Check the distribution of allele frequencies if `AF` is present in the INFO field.
```bash
bcftools query -f '%INFO/AF\n' your_file.vcf.gz | sort -n | uniq -c
```

#### **1.6 Genotype fields available**
View the genotype fields in the VCF file (e.g., GT, DS, GP).
```bash
bcftools view -h your_file.vcf.gz | grep "##FORMAT"
```

#### **1.7 Multiallelic variants**
Count the number of multiallelic variants.
```bash
bcftools view -H your_file.vcf.gz | awk '{if(length($5)>1)print}' | wc -l
```

#### **1.8 Sample information**
Check the first few sample IDs.
```bash
bcftools query -l your_file.vcf.gz | head
```

#### **1.9 Presence of sex chromosomes**
Confirm the presence and naming convention of sex chromosomes.
```bash
bcftools view -H your_file.vcf.gz | cut -f1 | sort | uniq
```

#### **1.10 File compression and indexing**
Ensure the file is properly compressed and indexed.
```bash
file your_file.vcf.gz
bcftools index -s your_file.vcf.gz
```

#### **1.11 Genotype call rate**
Check the genotype call rate to assess missingness.
```bash
bcftools stats your_file.vcf.gz | grep "number of genotypes"
```

---

### 🔨 **2. Handling Multiallelic Variants**

#### **2.1 Split multiallelic variants**
Split multiallelic variants into multiple rows.
```bash
bcftools norm -m-any --check-ref w -f reference.fa input.vcf.gz -Oz -o output_split.vcf.gz
```
Or, using PLINK2:
```bash
plink2 --vcf input.vcf.gz --max-alleles 2 --make-pgen --out output_split
```

#### **2.2 Keep only biallelic SNPs**
Filter out multiallelic variants, retaining only biallelic SNPs.
```bash
bcftools view -m2 -M2 -v snps input.vcf.gz -Oz -o output_biallelic_snps.vcf.gz
```
Or, using PLINK2:
```bash
plink2 --vcf input.vcf.gz --max-alleles 2 --snps-only --make-bed --out output_biallelic_snps
```

#### **2.3 Keep the most common alternative allele**
Retain only the most common alternative allele.
```bash
bcftools norm -m+any --check-ref w -f reference.fa input.vcf.gz | \
bcftools view -m2 -M2 -Oz -o output_common_alt.vcf.gz
```

#### **2.4 Use PLINK2 PGEN format (multiallelic-friendly)**
Convert to PGEN format, which supports multiallelic variants.
```bash
plink2 --vcf input.vcf.gz --make-pgen --out output_multi
```

#### **2.5 Separate multiallelic variants for special analysis**
Keep multiallelic variants in a separate file.
```bash
bcftools view -m3 input.vcf.gz -Oz -o multiallelic_variants.vcf.gz
```

---

### 🔧 **3. VCF to PLINK Conversion**

#### **3.1 Original PLINK2 command**
Your initial pipeline command:
```bash
plink2 --vcf ${_input} \
    --vcf-half-call m \
    --vcf-require-gt \
    --allow-extra-chr \
    ${('--keep %s' % keep_samples) if keep_samples.is_file() else ""} \
    ${('--remove %s' % remove_samples) if remove_samples.is_file() else ""} \
    --make-bed --out ${_output:n}  \
    ${"--rm-dup exclude-all" if remove_duplicates else "" } \
    --threads ${numThreads} \
    --memory ${int(expand_size(mem) * 0.9)/1e06} \
    ${"--output-chr chrMT" if add_chr else ""}  
```

#### ✅ **3.2 Improved PLINK2 command**
You modified the pipeline for better handling of dosage, multiallelic variants, and sex chromosomes:
```bash
plink2 --vcf /home/rl3328/motor_data/geno_data/chrAll_GRCh38.vcf.gz \
    --vcf-half-call m \
    --import-dosage-certainty 0 \         # Preserve dosage information
    --not-chr X \                         # Exclude chrX
    --allow-extra-chr \                   # Allow non-standard chr names
    --make-bed \
    --set-all-var-ids '@:#:$r:$a' \       # Split multiallelic variants
    --max-alleles 2 \                      # Retain only biallelic variants
    --keep /home/rl3328/motor_data/geno_data/ROSMAP_sample_list.txt \
    --out /home/rl3328/motor_data/geno_data/chrAll_GRCh38 \
    --threads 20 \
    --memory 14400.0 --output-chr chrMT
```

---

### ✅ **Key Takeaways**

- You thoroughly inspect the VCF file before conversion, checking for multiallelic variants, dosage information, and non-standard chromosomes.
- You optimize your PLINK2 command by:
  - Preserving dosage information.
  - Excluding chrX.(sex information)
  - Allowing extra chromosome names.
  - Splitting multiallelic variants.
  - Retaining only biallelic variants.
  - Utilizing multi-threading and memory efficiently.

---
